# JN10 — Building Nested Bathymetric Grids with `nest_down`
## Hands-On: Cádiz & Huelva — Atlantic Domain (4-Level, Branching Hierarchy)

**HySEALab · Preprocessing Notebooks · EDANYA Research Group, Universidad de Málaga**
*Edited by José Manuel González Vida*

In this notebook you will work with the **Cádiz / Huelva** dataset, which has
a **branching** nested-grid structure: one L1 parent grid feeds **two** independent
L2 sub-grids (one for Cádiz, one for Huelva), each with its own L3.

---

### Requirements

**Python packages** (any recent version):

```bash
conda install -c conda-forge numpy scipy matplotlib netcdf4
# or: pip install numpy scipy matplotlib netCDF4
```

**Input data — Cádiz / Huelva grid dataset (included in the repository):**

The six bathymetric grids used in this notebook ship with the repository as
[`datasets/topobathy_JN10.zip`](../datasets/topobathy_JN10.zip) (~19 MB).
**The first code cell extracts it automatically** into `data/CA_HU/batimetrias/`
on first run — no manual step needed if you cloned the full repository.

```
preprocessing/
├── JN10_Nested_Grids_nest_down.ipynb
└── data/CA_HU/batimetrias/          (auto-extracted on first run)
    ├── GC_640m_L0.grd               L0  ~660 m  Atlantic basin
    ├── GC_160m_L1.grd               L1  ~165 m  Gulf of Cádiz
    ├── GC_CA01_40m_L2.grd           L2a ~41 m   Cádiz area
    ├── GC_CA01_Cadiz_10m_L3.grd     L3a ~10 m   Cádiz city
    ├── GC_HU01_40m_L2.grd           L2b ~41 m   Huelva area
    └── GC_HU01_PuntaUmbria_10m_L3.grd  L3b ~10 m  Punta Umbría
```

> The reference parameter file `ATL_GC_NC4_4L.txt` shown in Step 6 is **not**
> included; that step prints it only if you place a copy at
> `data/CA_HU/ATL_GC_NC4_4L.txt`, and is skipped gracefully otherwise.
> See [JN13](JN13_Parameter_File_Builder_and_Validator.ipynb) for the full
> parameter-file format.

> 💡 `nest_down` itself is **general**: to nest grids for your own study area
> you only need a parent `.grd` (e.g. built with [JN04](JN04_Grid_from_GEBCO.ipynb))
> and the bounding box of the child.

---

### Grid Hierarchy

```
L0  GC_640m_L0.grd       589 × 1265   ~660 m   Atlantic basin
│
└─► L1  GC_160m_L1.grd      1020 × 1560  ~165 m   Gulf of Cádiz
     │
     ├─► L2a  GC_CA01_40m_L2.grd        1252 × 1024   ~41 m   Cádiz area
     │    └─► L3a  GC_CA01_Cadiz_10m_L3.grd   1044 × 1060   ~10 m   Cádiz city
     │
     └─► L2b  GC_HU01_40m_L2.grd         708 × 1788   ~41 m   Huelva area
          └─► L3b  GC_HU01_PuntaUmbria_10m_L3.grd   612 × 700   ~10 m   Punta Umbría
```

### Learning Objectives

1. Understand a **branching** nested-grid hierarchy
2. Verify grid alignment for each parent → child pair
3. Use `nest_down` to create a new L2 grid for Cádiz
4. (Exercise) Create a new L2 grid for Huelva


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Libraries and data paths
# ─────────────────────────────────────────────────────────────────────────────

import itertools
import time
import os

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from netCDF4 import Dataset
from matplotlib import path as mplpath
from scipy.spatial import ConvexHull
from scipy.interpolate import RegularGridInterpolator
from datetime import datetime

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

# ── Paths to the Cádiz / Huelva bathymetric grids ────────────────────────────
# The dataset ships with the repository as datasets/topobathy_JN10.zip and is
# extracted automatically into data/CA_HU/batimetrias/ on first run.
DATA_DIR    = os.path.join('data', 'CA_HU')
BATHY_DIR   = os.path.join(DATA_DIR, 'batimetrias')
DATASET_ZIP = os.path.join('..', 'datasets', 'topobathy_JN10.zip')

if not os.path.isdir(BATHY_DIR):
    if os.path.isfile(DATASET_ZIP):
        import zipfile
        print(f'Extracting {DATASET_ZIP} -> {BATHY_DIR} ...')
        os.makedirs(BATHY_DIR, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP) as zf:
            zf.extractall(BATHY_DIR)
        print('Extraction done.')
        print()

# Shared L0 and L1
GRID0  = os.path.join(BATHY_DIR, 'GC_640m_L0.grd')             # L0 ~660 m  Atlantic basin
GRID1  = os.path.join(BATHY_DIR, 'GC_160m_L1.grd')             # L1 ~165 m  Gulf of Cádiz

# Cádiz branch
L2_CA  = os.path.join(BATHY_DIR, 'GC_CA01_40m_L2.grd')         # L2a ~41 m  Cádiz
L3_CA  = os.path.join(BATHY_DIR, 'GC_CA01_Cadiz_10m_L3.grd')   # L3a ~10 m  Cádiz city

# Huelva branch
L2_HU  = os.path.join(BATHY_DIR, 'GC_HU01_40m_L2.grd')         # L2b ~41 m  Huelva
L3_HU  = os.path.join(BATHY_DIR, 'GC_HU01_PuntaUmbria_10m_L3.grd')  # L3b ~10 m  Punta Umbría

# Output grids we will create with nest_down
MY_L2_CA = os.path.join(BATHY_DIR, 'my_L2_CA.grd')
MY_L2_HU = os.path.join(BATHY_DIR, 'my_L2_HU.grd')

# Verify that source files exist
print('SOURCE GRID FILES')
print('=' * 65)
for label, fpath in [
        ('L0  GC_640m_L0',              GRID0),
        ('L1  GC_160m_L1',              GRID1),
        ('L2a GC_CA01_40m_L2',          L2_CA),
        ('L3a GC_CA01_Cadiz_10m_L3',    L3_CA),
        ('L2b GC_HU01_40m_L2',          L2_HU),
        ('L3b GC_HU01_PuntaUmbria_10m', L3_HU),
]:
    exists = os.path.exists(fpath)
    marker = '✅' if exists else '❌'
    print(f'{marker}  {label:<30s}  {fpath}')

print()
if not os.path.isdir(BATHY_DIR):
    print('❌  Data folder not found:', os.path.abspath(BATHY_DIR))
    print('    Expected the dataset zip at', os.path.abspath(DATASET_ZIP))
    print('    (it ships with the repository — re-clone or download it from GitHub).')
    print()
print('Libraries loaded OK')
print(f'Working directory: {os.getcwd()}')


---
## Background: Branching vs Linear Hierarchies

In the simplest configurations the nesting is a **linear chain**: L0 → L1 → L2 → L3.

Here the hierarchy **branches**: from the same L1 Gulf-of-Cádiz grid, we zoom
into **two** separate coastal areas simultaneously:

```
                  ┌─► L2a Cádiz   ──► L3a Cádiz city
 L0 ──► L1 ───────┤
                  └─► L2b Huelva  ──► L3b Punta Umbría
```

Both L2 grids share the **same L1 parent**, so the alignment rule applies
separately for each:

$$
(b - x_{0,\text{parent}} + \tfrac{\Delta x_{\text{parent}}}{2}) \mod \Delta x_{\text{parent}} = 0
$$

where $b$ is any child boundary (west / east / south / north).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Utility functions: grdread and plot helpers
# ─────────────────────────────────────────────────────────────────────────────

def grdread(grdfile):
    """
    Read a HySEA .grd (NetCDF4) file and return coordinate vectors and the data matrix.

    Returns
    -------
    [x, y, z] : list of arrays
    dx : float  — longitude cell spacing (degrees)
    dy : float  — latitude  cell spacing (degrees)
    """
    ds        = Dataset(grdfile)
    var_names = list(ds.variables.keys())
    ejex = ['lon', 'x', 'longitude']
    ejey = ['lat', 'y', 'latitude']
    ejez = ['topo', 'z', 'Band1']
    x = y = z = None
    for name in var_names:
        if name in ejex: x = ds[name][:]
        if name in ejey: y = ds[name][:]
        if name in ejez: z = ds[name][:]
    ds.close()
    dx = float(x[1] - x[0])
    dy = float(y[1] - y[0])
    return [x, y, z], dx, dy


def print_grid_info(label, grdfile):
    """Print metadata summary for a .grd file."""
    (x, y, z), dx, dy = grdread(grdfile)
    dx_arcsec = dx * 3600
    dx_m      = dx * 111320
    print(f'  {label}')
    print(f'    Size     : {len(x)} × {len(y)} cells  ({len(x)*len(y)/1e6:.2f} M cells)')
    print(f'    Lon      : [{float(x[0]):.4f}, {float(x[-1]):.4f}] °E')
    print(f'    Lat      : [{float(y[0]):.4f}, {float(y[-1]):.4f}] °N')
    print(f'    dx       : {dx_arcsec:.2f}" ≈ {dx_m:.0f} m')
    print(f'    z range  : [{float(z.min()):.1f}, {float(z.max()):.1f}] m')
    print()


def plot_grid(ax, grdfile, title=None, vmin=-3000, vmax=500, cmap='terrain',
              show_colorbar=True, fig=None):
    """Plot a .grd bathymetry on a given Axes object."""
    (x, y, z), dx, dy = grdread(grdfile)
    dx_m = dx * 111320
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad('white', alpha=0.0)
    im = ax.pcolormesh(x, y, z, cmap=cm, shading='auto', vmin=vmin, vmax=vmax)
    ax.contour(x, y, z, levels=[0], colors='black', linewidths=0.7, alpha=0.8)
    if show_colorbar and fig is not None:
        fig.colorbar(im, ax=ax, pad=0.02, label='Depth / Elevation (m)')
    if title is None:
        title = f'{os.path.basename(grdfile)}  ({len(x)}×{len(y)}, Δx≈{dx_m:.0f} m)'
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Longitude (°E)', fontsize=8)
    ax.set_ylabel('Latitude (°N)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
    return im


print('Utility functions defined OK')


---
## Step 1 — Explore the Branching Grid Hierarchy

Let's read all six grids and inspect their metadata.  Pay attention to:

- The **refinement ratio** at each parent → child step (should be 4×)
- That **both** L2a (Cádiz) and L2b (Huelva) use the **same L1 parent**


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Inspect and compare all six grids
# ─────────────────────────────────────────────────────────────────────────────

print('GRID HIERARCHY — CÁDIZ / HUELVA (ATLANTIC)')
print('=' * 65)

print('  ── Shared trunk ──')
print_grid_info('L0  GC_640m_L0',     GRID0)
print_grid_info('L1  GC_160m_L1',     GRID1)

print('  ── Cádiz branch ──')
print_grid_info('L2a GC_CA01_40m_L2',       L2_CA)
print_grid_info('L3a GC_CA01_Cadiz_10m_L3', L3_CA)

print('  ── Huelva branch ──')
print_grid_info('L2b GC_HU01_40m_L2',              L2_HU)
print_grid_info('L3b GC_HU01_PuntaUmbria_10m_L3',  L3_HU)

# ── Refinement ratios ──────────────────────────────────────────────────────
_, dx0, _ = grdread(GRID0)
_, dx1, _ = grdread(GRID1)
_, dx2ca, _ = grdread(L2_CA)
_, dx2hu, _ = grdread(L2_HU)
_, dx3ca, _ = grdread(L3_CA)
_, dx3hu, _ = grdread(L3_HU)

print('REFINEMENT RATIOS')
print(f'  L0  → L1      : {dx0/dx1:.2f}  (nominal = 4)')
print(f'  L1  → L2a CA  : {dx1/dx2ca:.2f}  (nominal = 4)')
print(f'  L1  → L2b HU  : {dx1/dx2hu:.2f}  (nominal = 4)')
print(f'  L2a → L3a CA  : {dx2ca/dx3ca:.2f}  (nominal = 4)')
print(f'  L2b → L3b HU  : {dx2hu/dx3hu:.2f}  (nominal = 4)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Plot the branching hierarchy
# Panel 0: L0 (Gulf-of-Cádiz zoom) + L1 box
# Panel 1: L1 + L2a (Cádiz) box + L2b (Huelva) box
# Panel 2: L2a (Cádiz) + L3a box
# Panel 3: L2b (Huelva) + L3b box
# ─────────────────────────────────────────────────────────────────────────────

(x0, y0, z0), dx0, dy0 = grdread(GRID0)
(x1, y1, z1), dx1, dy1 = grdread(GRID1)
(x2ca, y2ca, z2ca), dx2ca, dy2ca = grdread(L2_CA)
(x2hu, y2hu, z2hu), dx2hu, dy2hu = grdread(L2_HU)
(x3ca, y3ca, z3ca), dx3ca, dy3ca = grdread(L3_CA)
(x3hu, y3hu, z3hu), dx3hu, dy3hu = grdread(L3_HU)

fig, axes = plt.subplots(1, 4, figsize=(20, 6), constrained_layout=True)
fig.suptitle('Cádiz / Huelva — 4-Level Branching Nested Grid Hierarchy', fontsize=13)

def draw_box(ax, x, y, color, label):
    dx_ = float(x[1]-x[0]); dy_ = float(y[1]-y[0])
    x0b = float(x[0]) - dx_/2;  x1b = float(x[-1]) + dx_/2
    y0b = float(y[0]) - dy_/2;  y1b = float(y[-1]) + dy_/2
    ax.plot([x0b,x1b,x1b,x0b,x0b],[y0b,y0b,y1b,y1b,y0b],
            '-', color=color, linewidth=2, label=label, zorder=5)

cm_ = plt.get_cmap('terrain').copy(); cm_.set_bad('white', 0.0)

# Panel 0: L0 zoomed to Gulf of Cádiz + L1 box
ax = axes[0]
mask_lon = (x0 >= -13.0) & (x0 <= -4.5)
mask_lat = (y0 >= 33.5) & (y0 <= 38.0)
ax.pcolormesh(x0[mask_lon], y0[mask_lat],
              z0[np.ix_(mask_lat, mask_lon)],
              cmap=cm_, shading='auto', vmin=-5000, vmax=500)
ax.contour(x0[mask_lon], y0[mask_lat],
           z0[np.ix_(mask_lat, mask_lon)],
           levels=[0], colors='black', linewidths=0.7)
draw_box(ax, x1, y1, 'gold', 'L1 domain')
ax.set_title('L0  GC_640m_L0.grd\n(~660 m, Gulf-of-Cádiz window)', fontsize=9)
ax.set_xlabel('Lon (°E)', fontsize=8); ax.set_ylabel('Lat (°N)', fontsize=8)
ax.tick_params(labelsize=7)
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)

# Panel 1: L1 + both L2 boxes
ax = axes[1]
plot_grid(ax, GRID1, title='L1  GC_160m_L1.grd\n(~165 m)', show_colorbar=False,
          vmin=-3000, vmax=500)
draw_box(ax, x2ca, y2ca, 'tomato',      'L2a Cádiz')
draw_box(ax, x2hu, y2hu, 'deepskyblue', 'L2b Huelva')
ax.legend(fontsize=7, loc='lower right')

# Panel 2: L2a (Cádiz) + L3a box
ax = axes[2]
plot_grid(ax, L2_CA, title='L2a  GC_CA01_40m_L2.grd\n(~41 m, Cádiz)',
          show_colorbar=False, vmin=-2000, vmax=500)
draw_box(ax, x3ca, y3ca, 'deepskyblue', 'L3a Cádiz city')
ax.legend(fontsize=7, loc='lower right')

# Panel 3: L2b (Huelva) + L3b box
ax = axes[3]
plot_grid(ax, L2_HU, title='L2b  GC_HU01_40m_L2.grd\n(~41 m, Huelva)',
          show_colorbar=False, vmin=-2000, vmax=500, fig=fig)
draw_box(ax, x3hu, y3hu, 'deepskyblue', 'L3b Punta Umbría')
ax.legend(fontsize=7, loc='lower right')

plt.show()
print('Note: L2a (Cádiz) and L2b (Huelva) are both nested inside the same L1 grid.')


---
## Step 2 — Verify the Alignment of the Existing Grids

A child grid is **correctly aligned** with its parent when every boundary of the
child (west / east / south / north) falls **exactly on a parent cell edge**.

### The alignment concept — a visual explanation

Parent cells have centres spaced $\Delta x$ apart. Their **edges** lie halfway
between those centres:

```
Parent cells (Δx = 1.0°, centres at …10.0  11.0  12.0  13.0…)

  edges:    9.5   10.5   11.5   12.5   13.5
             |      |      |      |      |
  cells:   [  10.0  ][  11.0  ][  12.0  ][  13.0  ]
```

A child boundary $b$ is aligned if it lands exactly on one of those edges.

| Child boundary | Distance to nearest edge | Aligned? |
|---|---|---|
| $b = 10.5°$ | 0.000° | ✅ Perfect |
| $b = 10.7°$ | 0.200° | ⚠️ Off by 0.2° |
| $b = 12.500000001°$ | 10⁻⁹° | ✅ OK (floating-point noise) |

### How the residual is computed

We measure the **distance from $b$ to the nearest parent cell edge**:

1. Shift $b$ into the edge coordinate system: $b' = b - (x_0 - \Delta x / 2)$
   where $x_0$ is the western edge of the first parent cell.
2. Find the remainder: $r = b' \bmod \Delta x$  (how far past the last edge).
3. Take the minimum of $r$ and $\Delta x - r$  (nearest edge, left or right).

$$r_\text{residual} = \min\!\left(b' \bmod \Delta x,\; \Delta x - b' \bmod \Delta x\right)$$

**Acceptance threshold:** $r < 10^{-6}$° ≈ 10 cm — well below any physical grid spacing.

Here we check all **five** parent → child pairs in the branching tree.

> ⚠️ **Note:** The reference grids in the training data were produced with external
> tools that do not enforce this rule, so you will see several ⚠️ CHECK flags below.
> **This is expected** — read the explanation cell after the output.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — Check alignment for all parent → child pairs
# ─────────────────────────────────────────────────────────────────────────────

TOL = 1e-6

def check_alignment(parent_file, child_file, parent_label, child_label):
    """Verify that a child .grd is correctly aligned with its parent."""
    (xp, yp, _), dxp, dyp = grdread(parent_file)
    (xc, yc, _), dxc, dyc = grdread(child_file)
    west  = float(xc[0])  - dxc / 2.0
    east  = float(xc[-1]) + dxc / 2.0
    south = float(yc[0])  - dyc / 2.0
    north = float(yc[-1]) + dyc / 2.0
    origin_x = float(xp[0]) - dxp / 2.0
    origin_y = float(yp[0]) - dyp / 2.0
    res = {}
    for name, b, step, orig in [
            ('West',  west,  dxp, origin_x),
            ('East',  east,  dxp, origin_x),
            ('South', south, dyp, origin_y),
            ('North', north, dyp, origin_y),
    ]:
        r = abs((b - orig) % step)
        r = min(r, step - r)
        res[name] = r
    ratio_x = round(dxp / dxc)
    ratio_y = round(dyp / dyc)
    print(f'  {parent_label} → {child_label}   ratio = {ratio_x} (x), {ratio_y} (y)')
    all_ok = True
    for name, r in res.items():
        status = 'OK ✅' if r < TOL else '⚠️  CHECK'
        if r >= TOL: all_ok = False
        print(f'    {name:<6}: residual = {r:.2e}  [{status}]')
    if all_ok:
        print('    ─ Perfect alignment ─')
    print()

print('ALIGNMENT CHECKS (existing grids)')
print('=' * 60)
check_alignment(GRID0, GRID1,  'L0 GC_640m',    'L1 GC_160m')
check_alignment(GRID1, L2_CA,  'L1 GC_160m',    'L2a CA01_40m')
check_alignment(GRID1, L2_HU,  'L1 GC_160m',    'L2b HU01_40m')
check_alignment(L2_CA, L3_CA,  'L2a CA01_40m',  'L3a Cadiz_10m')
check_alignment(L2_HU, L3_HU,  'L2b HU01_40m',  'L3b PuntaUmbria_10m')


---
### 📋 Understanding the ⚠️ CHECK Results Above

**The ⚠️ CHECK flags on the reference grids are expected — this is intentional.**

The grids supplied in the training data (`GC_640m_L0_nc4.grd`, `GC_160m_L1_nc4.grd`, etc.) were produced with external tools (GMT) that clip to requested lon/lat corners *without* snapping to parent cell edges. The typical result is that each boundary is displaced by **1–2 child cell widths** from the nearest parent edge.

| Parent → Child | Largest residual | ≈ Child cells displaced |
|---|---|---|
| L0 (660 m) → L1 (165 m) | 1.48 × 10⁻³° | ≈ 1 × dx_L1 |
| L1 (165 m) → L2-CA (41 m) | 7.40 × 10⁻⁴° | ≈ 2 × dx_L2 |
| L1 (165 m) → L2-HU (41 m) | 3.70 × 10⁻⁴° | ≈ 1 × dx_L2 |
| L2-CA (41 m) → L3-CA (10 m) | 1.84 × 10⁻⁴° | ≈ 2 × dx_L3 |
| L2-HU (41 m) → L3-HU (10 m) | 1.85 × 10⁻⁴° | ≈ 2 × dx_L3 |

#### Does misalignment affect the simulation?

A displacement of 1–2 child cells at the nesting boundary introduces a small **interpolation error** in the boundary conditions passed from parent to child. For tsunami wavelengths (10–500 km) this is negligible — the wave does not 'see' a 40 m offset. However, for inundation at L3 (10 m resolution) it can produce minor artificial reflections at the boundary and slight amplitude errors of a few percent.

> **For research-grade simulations, always use `nest_down()` to build your grids** — it guarantees alignment residuals < 10⁻⁶° (≈ 10 cm) at all four boundaries.  
> The reference grids here are provided for *visual inspection only*. In Steps 4–5 you will create a new grid (`my_L2_CA.grd`) with perfect alignment and verify it.


---
## Step 2b — Visualisation: Grid Cells and Cell Centres

The next cell produces a **microscopic zoom** over a small region of the domain
and draws for each level:

- The **rectangle** of each grid cell (blue = coarse grid, red = fine grid)
- A **dot** at the centre of each cell
- The **cell-edge lines** of the coarse grid (thick solid blue) as the alignment reference

If the grids are correctly aligned, the fine cell edges (dashed red) will coincide
exactly with the coarse cell edges (thick solid blue).

We run the zoom for the **L1 → L2a (Cádiz)** pair.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5b — Alignment zoom: grid cell rectangles + centre dots
# ─────────────────────────────────────────────────────────────────────────────

def plot_alignment_zoom(coarse_file, fine_file, n_coarse=4,
                        coarse_color='steelblue', fine_color='tomato',
                        title=None):
    """
    Microscopic zoom that draws coarse and fine grid cells with a dot at each
    cell centre, so the alignment between the two levels can be verified visually.

    Parameters
    ----------
    coarse_file  : str  — parent .grd file (alignment reference)
    fine_file    : str  — child .grd file
    n_coarse     : int  — number of coarse cells to show along each axis
    """
    (xc, yc, _), dxc, dyc = grdread(coarse_file)
    (xf, yf, _), dxf, dyf = grdread(fine_file)

    cx = float(xf[len(xf)//2]);  cy = float(yf[len(yf)//2])
    ic = int(np.searchsorted(xc, cx));  jc = int(np.searchsorted(yc, cy))
    h  = n_coarse // 2
    xc_z = np.array(xc[max(0, ic-h) : min(len(xc), ic+h)])
    yc_z = np.array(yc[max(0, jc-h) : min(len(yc), jc+h)])

    lon0 = float(xc_z[0])  - dxc/2;  lon1 = float(xc_z[-1]) + dxc/2
    lat0 = float(yc_z[0])  - dyc/2;  lat1 = float(yc_z[-1]) + dyc/2
    xf_z = np.array(xf[(xf >= lon0 - dxf) & (xf <= lon1 + dxf)])
    yf_z = np.array(yf[(yf >= lat0 - dyf) & (yf <= lat1 + dyf)])

    fig, ax = plt.subplots(figsize=(9, 8))

    # Coarse cell rectangles
    for xi in xc_z:
        for yi in yc_z:
            ax.add_patch(mpatches.Rectangle(
                (float(xi)-dxc/2, float(yi)-dyc/2), dxc, dyc,
                lw=2, edgecolor=coarse_color, facecolor=coarse_color, alpha=0.12))
    # Fine cell rectangles
    for xi in xf_z:
        for yi in yf_z:
            ax.add_patch(mpatches.Rectangle(
                (float(xi)-dxf/2, float(yi)-dyf/2), dxf, dyf,
                lw=0.5, edgecolor=fine_color, facecolor=fine_color, alpha=0.10))
    # Coarse cell edges (thick solid — alignment reference)
    for xi in xc_z:
        ax.axvline(float(xi)-dxc/2, color=coarse_color, lw=2.0, alpha=0.8, zorder=3)
    ax.axvline(float(xc_z[-1])+dxc/2, color=coarse_color, lw=2.0, alpha=0.8, zorder=3)
    for yi in yc_z:
        ax.axhline(float(yi)-dyc/2, color=coarse_color, lw=2.0, alpha=0.8, zorder=3)
    ax.axhline(float(yc_z[-1])+dyc/2, color=coarse_color, lw=2.0, alpha=0.8, zorder=3)
    # Fine cell edges (thin dashed)
    for xi in xf_z:
        ax.axvline(float(xi)-dxf/2, color=fine_color, lw=0.5, alpha=0.5, ls='--', zorder=2)
    ax.axvline(float(xf_z[-1])+dxf/2, color=fine_color, lw=0.5, alpha=0.5, ls='--', zorder=2)
    for yi in yf_z:
        ax.axhline(float(yi)-dyf/2, color=fine_color, lw=0.5, alpha=0.5, ls='--', zorder=2)
    ax.axhline(float(yf_z[-1])+dyf/2, color=fine_color, lw=0.5, alpha=0.5, ls='--', zorder=2)
    # Centre dots — COARSE
    XC, YC = np.meshgrid(xc_z, yc_z)
    ax.scatter(XC.ravel(), YC.ravel(), s=120, c=coarse_color, zorder=6,
               edgecolors='white', linewidths=0.8,
               label=f'Coarse centres  (dx≈{dxc*111320:.0f} m)')
    # Centre dots — FINE
    XF, YF = np.meshgrid(xf_z, yf_z)
    ax.scatter(XF.ravel(), YF.ravel(), s=20, c=fine_color, zorder=5,
               edgecolors='none',
               label=f'Fine centres  (dx≈{dxf*111320:.0f} m)')
    ax.set_xlim(lon0, lon1);  ax.set_ylim(lat0, lat1)
    ax.set_aspect('equal')
    ax.ticklabel_format(useOffset=False, style='plain')
    ax.set_xlabel('Longitude (°E)');  ax.set_ylabel('Latitude (°N)')
    ax.legend(fontsize=9, loc='upper right');  ax.grid(False)
    ratio_real = round(dxc / dxf)
    if title is None:
        title = (f'Alignment zoom — {os.path.basename(coarse_file)} → '
                 f'{os.path.basename(fine_file)}\n'
                 f'Ratio = {ratio_real}  |  {n_coarse}×{n_coarse} coarse cells shown')
    ax.set_title(title, fontsize=10)
    plt.tight_layout();  plt.show()
    print(f'Each coarse cell (blue) contains {ratio_real}×{ratio_real} = {ratio_real**2} fine cells (red).')
    print('Blue cell edges coincide with dashed red edges → alignment is correct.')


# ── Run the alignment zoom for the L1 → L2a (Cádiz) pair ─────────────────────
plot_alignment_zoom(GRID1, L2_CA, n_coarse=4)


---
## Step 3 — The `nest_down` Function

`nest_down` creates a new nested sub-grid that is:

1. **Aligned** with the parent (child boundaries on parent cell edges)
2. **Interpolated** from a fine-resolution source dataset
3. **Written** as a HySEA-compatible NetCDF4 `.grd` file

| Argument | Meaning |
|----------|---------|
| `finemeshfile`   | Source bathymetry at the desired fine resolution |
| `coarsemeshfile` | Parent grid — used only for alignment |
| `ratio`          | Refinement factor (e.g. 4 → parent cell = 4 child cells) |
| `SW`, `NE`       | Desired output domain corners `[lon, lat]` |
| `foutput`        | Path for the output `.grd` file |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Define grdwrite and nest_down
# ─────────────────────────────────────────────────────────────────────────────

def grdwrite(x, y, z, foutput):
    """
    Write arrays x, y, z to a HySEA-compatible NetCDF4 .grd file.
    """
    today   = datetime.today()
    dataset = Dataset(foutput, 'w', format='NETCDF4')
    dataset.createDimension('x', len(x))
    dataset.createDimension('y', len(y))
    lon_var = dataset.createVariable('x', 'f8', 'x')
    lat_var = dataset.createVariable('y', 'f8', 'y')
    z_var   = dataset.createVariable('z', 'f4', ('y', 'x'))
    lon_var[:] = x;  lat_var[:] = y;  z_var[:,:] = z
    dataset.Conventions  = 'CF-1.6'
    dataset.title        = os.path.basename(foutput)
    dataset.history      = 'File written using netCDF4 Python module'
    dataset.description  = 'Created ' + today.strftime('%d/%m/%Y')
    dataset.GMT_version  = '6.1.0'
    lon_var.units = 'degrees_east'
    lat_var.units = 'degrees_north'
    z_var.units   = 'meters'
    dataset.close()


def nest_down(finemeshfile, coarsemeshfile, ratio, SW, NE, foutput):
    """
    Create a high-resolution nested .grd file aligned with a coarser parent grid.

    Parameters
    ----------
    finemeshfile   : str   — fine-resolution source data
    coarsemeshfile : str   — parent grid (alignment reference only)
    ratio          : int   — refinement ratio (e.g. 4)
    SW             : list  — [lon, lat] south-west corner of output domain
    NE             : list  — [lon, lat] north-east corner of output domain
    foutput        : str   — output .grd file path
    """
    class G: pass
    class F: pass

    G.x, G.y, G.Z = grdread(coarsemeshfile)[0]
    dx_grosera     = grdread(coarsemeshfile)[1]
    G.X, G.Y       = np.meshgrid(G.x, G.y)

    F.x, F.y, F.Z  = grdread(finemeshfile)[0]
    dx_fina        = grdread(finemeshfile)[1]
    dy_fina        = grdread(finemeshfile)[2]
    F.X, F.Y       = np.meshgrid(F.x, F.y)

    u1   = np.ma.compress_cols(F.X).flatten()
    v1   = np.ma.compress_cols(F.Y).flatten()
    pts  = np.vstack([u1, v1]).T
    hull = ConvexHull(pts)
    k    = hull.vertices
    pts_mod = pts.copy()
    pts_mod[k[0]] += np.array([-dx_fina/2, -dy_fina/2])
    pts_mod[k[1]] += np.array([ dx_fina/2, -dy_fina/2])
    pts_mod[k[2]] += np.array([ dx_fina/2,  dy_fina/2])
    pts_mod[k[3]] += np.array([-dx_fina/2,  dy_fina/2])
    LLB = pts_mod[k[0]];  URB = pts_mod[k[2]]

    SW_arr = np.array(SW);  NE_arr = np.array(NE)
    if SW_arr[0] < LLB[0] or SW_arr[1] < LLB[1] or \
       NE_arr[0] > URB[0] or NE_arr[1] > URB[1]:
        print('⚠️  WARNING: requested domain may extend outside the fine source — NaN values possible.')

    SE_arr = np.array([NE_arr[0], SW_arr[1]])
    NW_arr = np.array([SW_arr[0], NE_arr[1]])

    pts_coarse = np.array(list(itertools.product(G.x, G.y)))
    box    = mplpath.Path([SW_arr, SE_arr, NE_arr, NW_arr])
    inside = box.contains_points(pts_coarse)
    Gx_flat = G.X.T.flatten();  Gy_flat = G.Y.T.flatten()
    Ax = np.unique(Gx_flat[inside])
    Ay = np.unique(Gy_flat[inside])
    dx = dx_grosera
    dy = float(Ay[1] - Ay[0])

    if (Ax[0]  - dx/2.) < LLB[0]: Ax = Ax[1:]
    if (Ax[-1] + dx/2.) > URB[0]: Ax = Ax[:-1]
    if (Ay[0]  - dy/2.) < LLB[1]: Ay = Ay[1:]
    if (Ay[-1] + dy/2.) > URB[1]: Ay = Ay[:-1]

    xx_new = np.arange(Ax[0]  - dx/2 + dx/(2*ratio),
                       Ax[-1] + dx/2 - dx/(4*ratio),
                       dx / ratio)
    yy_new = np.arange(Ay[0]  - dy/2 + dy/(2*ratio),
                       Ay[-1] + dy/2 - dy/(4*ratio),
                       dy / ratio)

    print(f'Interpolating onto {len(xx_new)} × {len(yy_new)} output grid...')
    print('(this may take a minute for large grids)')
    valores = F.Z.T
    interp  = RegularGridInterpolator((F.x, F.y), valores)
    new_points = np.array(list(itertools.product(xx_new, yy_new)))
    Zc = interp(new_points).reshape(len(xx_new), len(yy_new)).T

    grdwrite(xx_new, yy_new, Zc, foutput)
    print(f'✅  New grid written: {foutput}')
    print(f'   Dimensions : {len(xx_new)} × {len(yy_new)} cells')
    print(f'   Lon range  : [{float(xx_new[0]):.4f}, {float(xx_new[-1]):.4f}] °E')
    print(f'   Lat range  : [{float(yy_new[0]):.4f}, {float(yy_new[-1]):.4f}] °N')
    dxout = float(xx_new[1] - xx_new[0])
    print(f'   Resolution : {dxout*3600:.2f}" ≈ {dxout*111320:.0f} m')
    return foutput


print('grdwrite and nest_down defined OK')


---
## Step 4 — Hands-On: Build a New L2 Grid for Cádiz

We will call `nest_down` to create `my_L2_CA.grd`:

| Parameter | Value | Why |
|-----------|-------|-----|
| `finemeshfile`   | `GC_CA01_40m_L2.grd` | Source bathymetry at ~41 m |
| `coarsemeshfile` | `GC_160m_L1.grd`     | Parent — used for alignment only |
| `ratio`          | 4                    | One L1 cell = 4 output cells |
| `SW`             | `[-6.45, 36.37]`     | SW corner of sub-domain |
| `NE`             | `[-6.12, 36.80]`     | NE corner of sub-domain |

> **Note:** `finemeshfile` provides the actual bathymetric values;
> `coarsemeshfile` only provides the alignment grid. The output resolution is
> `dx_parent / ratio = 165 / 4 ≈ 41 m`.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — Run nest_down to create my_L2_CA.grd
# ─────────────────────────────────────────────────────────────────────────────

FINE_SOURCE   = L2_CA    # GC_CA01_40m_L2.grd  (~41 m  source data)
COARSE_PARENT = GRID1    # GC_160m_L1.grd       (~165 m parent for alignment)
RATIO         = 4

# Sub-domain inside GC_CA01_40m_L2.grd coverage:
# lon=[-6.4752, -6.0963],  lat=[36.3548, 36.8164]
SW_DOMAIN = [-6.45, 36.37]   # [lon, lat] south-west
NE_DOMAIN = [-6.12, 36.80]   # [lon, lat] north-east

OUTPUT_FILE = MY_L2_CA   # '...batimetrias/my_L2_CA.grd'

print('CALLING nest_down')
print(f'  Fine source : {os.path.basename(FINE_SOURCE)}')
print(f'  Parent grid : {os.path.basename(COARSE_PARENT)}')
print(f'  Ratio       : {RATIO}')
print(f'  SW          : {SW_DOMAIN}')
print(f'  NE          : {NE_DOMAIN}')
print(f'  Output      : {OUTPUT_FILE}')
print()

t0 = time.time()
nest_down(FINE_SOURCE, COARSE_PARENT, RATIO, SW_DOMAIN, NE_DOMAIN, OUTPUT_FILE)
print(f'\nTime elapsed : {time.time()-t0:.1f} s')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Visualize my_L2_CA.grd and compare with source GC_CA01_40m_L2.grd
# ─────────────────────────────────────────────────────────────────────────────

(xnew, ynew, znew), dxnew, dynew = grdread(MY_L2_CA)

fig, axes = plt.subplots(1, 3, figsize=(17, 6), constrained_layout=True)
fig.suptitle('New L2 Grid for Cádiz vs Reference Data', fontsize=13)

# Panel 0: Parent L1 with output domain box
ax = axes[0]
plot_grid(ax, GRID1, title='Parent (L1)  GC_160m_L1.grd\n(~165 m)',
          show_colorbar=False, vmin=-3000, vmax=500)
x0b = float(xnew[0]) - dxnew/2;  x1b = float(xnew[-1]) + dxnew/2
y0b = float(ynew[0]) - dynew/2;  y1b = float(ynew[-1]) + dynew/2
ax.plot([x0b,x1b,x1b,x0b,x0b],[y0b,y0b,y1b,y1b,y0b],
        'tomato', linewidth=2, label='my_L2_CA extent')
ax.legend(fontsize=8)

# Panel 1: Our new grid
ax = axes[1]
cm_ = plt.get_cmap('terrain').copy(); cm_.set_bad('white', 0.0)
im  = ax.pcolormesh(xnew, ynew, znew, cmap=cm_, shading='auto', vmin=-2000, vmax=500)
ax.contour(xnew, ynew, znew, levels=[0], colors='black', linewidths=0.7)
ax.set_title(f'NEW  my_L2_CA.grd\n({len(xnew)}×{len(ynew)}, Δx≈{dxnew*111320:.0f} m)', fontsize=9)
ax.set_xlabel('Lon (°E)', fontsize=8); ax.set_ylabel('Lat (°N)', fontsize=8)
ax.tick_params(labelsize=7)
ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
fig.colorbar(im, ax=ax, pad=0.02, label='Depth / Elevation (m)')

# Panel 2: Reference source grid
ax = axes[2]
plot_grid(ax, L2_CA, title='Reference (source)  GC_CA01_40m_L2.grd\n(~41 m)',
          show_colorbar=False, vmin=-2000, vmax=500)
ax.plot([x0b,x1b,x1b,x0b,x0b],[y0b,y0b,y1b,y1b,y0b],
        'tomato', linewidth=2, label='my_L2_CA extent')
ax.legend(fontsize=8)

plt.show()

print('DEPTH STATISTICS COMPARISON')
print(f'  my_L2_CA.grd       : min={float(znew.min()):.1f} m,  max={float(znew.max()):.1f} m,  mean={float(znew.mean()):.1f} m')
(x2r, y2r, z2r), _, _ = grdread(L2_CA)
print(f'  GC_CA01_40m_L2.grd : min={float(z2r.min()):.1f} m,  max={float(z2r.max()):.1f} m,  mean={float(z2r.mean()):.1f} m')


---
## Step 5 — Verify Alignment of the New Grid

Before using `my_L2_CA.grd` in a simulation, we verify that:

1. It is **correctly aligned** with the L1 parent
2. It contains **no NaN values** (which would indicate the requested domain extended beyond the source data)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — Alignment verification for my_L2_CA.grd
# ─────────────────────────────────────────────────────────────────────────────

print('ALIGNMENT CHECK — New Grid vs Parent')
print('=' * 55)
check_alignment(GRID1, MY_L2_CA, 'L1 GC_160m (parent)', 'my_L2_CA (new)')

print('NaN CHECK')
(xout, yout, zout), _, _ = grdread(MY_L2_CA)
nan_count = int(np.sum(np.isnan(zout)))
if nan_count == 0:
    print('  ✅  No NaN values found — grid is ready for simulation')
else:
    print(f'  ❌  {nan_count} NaN values detected — check that GC_CA01_40m_L2.grd covers')
    print('     the entire SW → NE domain you requested.')


---
## Step 6 — The Parameter File: ATL_GC_NC4_4L.txt

The branching hierarchy is declared in the parameter file.  The key lines for
nesting are:

```
4              ← number of levels
4              ← refinement ratio L0 → L1
1              ← number of L1 grids
batimetrias/GC_160m_L1_nc4.grd
...
4              ← refinement ratio L1 → L2
2              ← number of L2 grids  ← TWO child grids from the same parent!
batimetrias/GC_HU01_40m_L2_nc4.grd
...
batimetrias/GC_CA01_40m_L2_nc4.grd
...
```

To replace the reference L2 Cádiz grid with your new `my_L2_CA.grd`, you would
update the corresponding line in the parameter file and point to the nc4 version.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10 — Display ATL_GC_NC4_4L.txt for reference
# ─────────────────────────────────────────────────────────────────────────────

parfile = os.path.join(DATA_DIR, 'ATL_GC_NC4_4L.txt')

if os.path.exists(parfile):
    print('PARAMETER FILE:', parfile)
    print('=' * 70)
    with open(parfile) as f:
        for i, line in enumerate(f, 1):
            print(f'{i:3d}  {line}', end='')
else:
    print(f'File not found: {parfile}')


---
## Exercise — Build a New L2 Grid for Huelva

Repeat the same workflow for the **Huelva branch**.

The source grid `GC_HU01_40m_L2.grd` covers:
- `lon = [-7.3701, -6.7082]`
- `lat = [37.0558, 37.3172]`

Choose a sub-domain inside this coverage, call `nest_down` with:
- `finemeshfile   = L2_HU`
- `coarsemeshfile = GRID1`   (same parent as Cádiz!)
- `ratio          = 4`
- Output          → `MY_L2_HU`

After creating the grid, run `check_alignment` and the NaN check.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11 — Exercise: build my_L2_HU.grd for Huelva
# Adjust SW and NE as needed to explore different sub-domains
# ─────────────────────────────────────────────────────────────────────────────

# ── Your configuration ────────────────────────────────────────────────────────
SW_HU = [-7.35, 37.07]   # ← try adjusting these values
NE_HU = [-6.73, 37.31]

# ── Call nest_down ────────────────────────────────────────────────────────────
t0 = time.time()
nest_down(
    finemeshfile   = L2_HU,
    coarsemeshfile = GRID1,
    ratio          = 4,
    SW             = SW_HU,
    NE             = NE_HU,
    foutput        = MY_L2_HU
)
print(f'Time: {time.time()-t0:.1f} s')

# ── Alignment check ───────────────────────────────────────────────────────────
print()
check_alignment(GRID1, MY_L2_HU, 'L1 GC_160m (parent)', 'my_L2_HU (new)')

# ── NaN check ─────────────────────────────────────────────────────────────────
print('NaN CHECK')
(xhu, yhu, zhu), _, _ = grdread(MY_L2_HU)
nan_count_hu = int(np.sum(np.isnan(zhu)))
if nan_count_hu == 0:
    print('  ✅  No NaN values — Huelva grid is ready')
else:
    print(f'  ❌  {nan_count_hu} NaN values — check domain coverage')

# ── Quick plot ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
fig.suptitle('Exercise: New L2 Grid for Huelva', fontsize=12)
plot_grid(axes[0], GRID1, title='Parent: GC_160m_L1 (~165 m)',
          show_colorbar=False, vmin=-3000, vmax=500)
(xhu_n, yhu_n, zhu_n), dxhu, dyhu = grdread(MY_L2_HU)
x0b = float(xhu_n[0]) - dxhu/2;  x1b = float(xhu_n[-1]) + dxhu/2
y0b = float(yhu_n[0]) - dyhu/2;  y1b = float(yhu_n[-1]) + dyhu/2
axes[0].plot([x0b,x1b,x1b,x0b,x0b],[y0b,y0b,y1b,y1b,y0b],'tomato',lw=2,label='my_L2_HU')
axes[0].legend(fontsize=8)
plot_grid(axes[1], MY_L2_HU,
          title=f'my_L2_HU ({len(xhu_n)}×{len(yhu_n)}, Δx≈{dxhu*111320:.0f} m)',
          show_colorbar=False, vmin=-3000, vmax=500, fig=fig)
plt.show()


---
## Summary

In this notebook you have:

| Step | What you did |
|------|--------------|
| 1 | Inspected a **branching** 4-level hierarchy (L0 → L1 → L2a/L2b → L3a/L3b) |
| 2 | Verified alignment for all **five** parent → child pairs |
| 2b | Visualised cell alignment microscopically (rectangles + centre dots) |
| 3 | Reviewed the `nest_down` function and `grdwrite` |
| 4 | Built `my_L2_CA.grd` for Cádiz using `nest_down` |
| 5 | Confirmed alignment and NaN-free output |
| 6 | Read the branching parameter file `ATL_GC_NC4_4L.txt` |
| Ex | Built `my_L2_HU.grd` for Huelva (parallel branch) |

**Key feature of this case:** the hierarchy **branches** — two L2 grids share
the same L1 parent.  `nest_down` handles this transparently; you simply call
it once per child with the same `coarsemeshfile = GRID1`.

➡ To validate a parameter file that declares this branching hierarchy, see
**[JN13](JN13_Parameter_File_Builder_and_Validator.ipynb)** — its hierarchy
viewer draws the L0 → L1 → L2a/L2b → L3a/L3b tree automatically.
